# Peeling Back the Curtain on AI Agents: 

When you build a reasoning assistant today, frameworks make it look incredibly clean.

- You import a prebuilt agent wrapper 
- pass it a model
- pass a list of functions


suddenly you have a system capable of multi-step logical cycles.

But hiding behind that convenience is a complex dance of message arrays, state
mutations, and API abstractions. If you only ever use the high-level wrappers, you are
treating your agent architecture like a black box.

In this deep dive, we are going to reverse-engineer the popular ReAct (Reasoning and
Acting) Agent pattern. We will start with the modern, high-level approach, peel it back
to see how graphs are explicitly structured, and finally build the entire machine
completely from scratch using standard Python loops.

## Level 1: The Modern Magic (create_agent)

In current production environments, you can implement a full ReAct loop in just a few
lines of code. By leveraging LangGraph's prebuilt architectures and the MessagesState
schema, the framework hides the wiring completely.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent

In [2]:

@tool
def add(a: int, b: int) -> int:
    """This tool adds two numbers together."""
    return a + b

model = ChatOllama(model="qwen2.5:3b")
system_prompt = "You are a precise, tool-using mathematical assistant."

app = create_agent(
    model=model,
    tools=[add]
)

inputs = {"messages": [("user", "add 3 + 4.")]}
for update in app.stream(inputs, stream_mode="updates"):
    print(update)

{'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-11-02T16:31:33.334909661Z', 'done': True, 'done_reason': 'stop', 'total_duration': 185720168685, 'load_duration': 7337385664, 'prompt_eval_count': 170, 'prompt_eval_duration': 143519278787, 'eval_count': 25, 'eval_duration': 34858222161, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a19d17-3a5b-7652-bf5c-b9faba3c9a2d-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'b760af8f-1708-4627-8c17-a298585cb61b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 25, 'total_tokens': 195})]}}
{'tools': {'messages': [ToolMessage(content='7', name='add', id='c4c7167a-459d-4f8d-a863-5a00c4ef4f9c', tool_call_id='b760af8f-1708-4627-8c17-a298585cb61b')]}}
{'model': {'messages': [AIMessage(content='The result of adding 3 and 4 is 7.', additional_kwargs={}, response_meta

# What is this abstracting away?

When you invoke create_react_agent, a massive amount of hidden orchestration
occurs under the hood:

- **Automatic Binding:** It inspects your @tool decorators, parses their signatures,
and calls .bind_tools() to generate the precise JSON schema your local model
requires.

- **Implicit State Management:** It swaps in MessagesState, which contains a
specialized "reducer" function (add_messages) that automatically appends new
items to your context history without destroying previous records.

- **Invisible Routing:** It injects a pre-written conditional block that intercepts the
model's output, checks for re

# Level 2: Peeling Back to the Explicit Graph Configuration

To see how those components connect, we need to peel off the first layer. Instead of a
prebuilt function, we can construct the graph canvas manually. This intermediateapproach reveals how separate processing nodes and conditional edges govern the
agent’s execution lifecycle.

In [3]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]



@tool
def add(a: int, b:int) -> int:
    '''this tool adds to numbers'''
    return a + b


tools = [add]
model = ChatOllama(model="qwen2.5:3b").bind_tools(tools=tools)


def model_call(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content= "You are my AI assistant, please answer my query to the best of your ability.")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

 
def should_contuinue(state:AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"


def save_mermaid(app):
    # Save the mermaid diagram directly as a PNG file
    with open("graph_diagram.png", "wb") as f:
        f.write(app.get_graph().draw_mermaid_png())

    print("Graph diagram saved successfully as 'graph_diagram.png'!")


graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)

tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node) 

graph.set_entry_point("our_agent")

graph.add_conditional_edges(
    "our_agent",
    should_contuinue,
    {
        "continue": "tools",
        "end": END,
    }
)

graph.add_edge("tools", "our_agent")
app = graph.compile()


def print_stream(stream):
    for s in stream:
        # values mode streams the entire state dict under the node name key
        for node_name, state in s.items():
            if "messages" in state and state["messages"]:
                message = state["messages"][-1]
                if isinstance(message, tuple):
                    print(message)
                else:
                    message.pretty_print()

inputs = {"messages": [("user", "add 3 + 4.")]}
print_stream(app.stream(inputs, stream_mode="updates"))
save_mermaid(app) 

================================== Ai Message ==================================
Tool Calls:
  add (f36bc8be-0437-4058-8555-e24813bee325)
 Call ID: f36bc8be-0437-4058-8555-e24813bee325
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: add

7
================================== Ai Message ==================================

The sum of 3 and 4 is 7. Is there anything else you'd like to know?
Graph diagram saved successfully as 'graph_diagram.png'!


## What does this layer show us?

By setting up the graph manually, we can observe the cyclical path:

1.​ model_call runs and generates an output.

2.​ should_continue looks at that output. If it finds .tool_calls, it vectors control
sideways to the execution node.

3.​ The ToolNode performs the math, outputs the outcome, and forces an absolute
edge traversal back to the reasoning node.


However, even this graph setup hides critical code. How does the model parse its
arguments? How do we map strings back to Python expressions? How does memory
behave without framework nodes?

# Level 3: The React Blueprint From Scratch

To completely deconstruct the agent, we must discard graphs, states, and framework
node configurations entirely.

Below is the complete brakedown of the ReAct architecture built utilizing a basic native Python dictionary for state management, a standard for loop to drive the cycle, and basic message data

containers (HumanMessage, AIMessage, SystemMessage) simply to hold text strings
cleanly.


## Part 1: Schema Compilation (Deconstructing .bind_tools())

In standard frameworks, developers call model.bind_tools([add, multiply]). Under the hood, this function relies on Python reflection (inspecting function signatures and docstrings) to compile native Python code into raw JSON configuration objects.


In [ ]:
tools_schema = [
    {
        "name": "add",
        "description": "Adds two integers together.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"}
            },
            "required": ["a", "b"]
        }
    },
    {
        "name": "multiply",
        "description": "Multiplies two integers together.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"}
            },
            "required": ["a", "b"]
        }
    }
]

• The Abstraction: Frameworks hide the target JSON grammar required by the model provider.
• The Reality: Passing this explicit array via .bind_tools() formats an open-source schema standard that tells your local Qwen instance exactly what input types (integer, string, etc.) and variable parameters are required to trigger a structured payload instead of conversational text.

Part 2: The Agent Node (The Pure Reasoning Step)

The core reasoning mechanism of an agent does not require special orchestration nodes or canvas modules. It is a stateless function that bridges network inputs with state updates.

In [ ]:
def agent_node(state: Dict[str, Any]) -> Dict[str, Any]:
    print("\n--- [NODE] Agent Thinking ---")
    messages = state["messages"]
    response = llm.invoke(messages)
    state["messages"].append(response)
    return state


• The Abstraction: Framework graph runtime engines wrap execution blocks inside abstract classes or complex node-handlers.
• The Reality: The engine simply extracts an array of structural message objects from a centralized dictionary wrapper (state["messages"]), forwards the context window across a local socket connection via llm.invoke(), captures the raw response token package, and appends it to the history log.

Part 3: The Conditional Router (The Decision Boundary)

Once an agent finishes an execution turn, it needs to evaluate what action to take next. Modern platforms handle this implicitly by reading message properties.

In [ ]:
def conditional_router(state: Dict[str, Any]) -> str:
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print("-> Router Decision: Tool calls detected. Routing to TOOL NODE.")
        return "call_tools"
    
    print("-> Router Decision: Final answer reached. Routing to END.")
    return "end"


• The Abstraction: Libraries like LangGraph use prepackaged edge targets like tools_condition to route traffic behind the scenes.
• The Reality: The router is nothing more than a basic Python if/else block inspecting the final array offset (state["messages"][-1]). If the provider's underlying protocol layer detected tool arguments, it populates a clean, normalized array inside .tool_calls. If that array has elements, we branch to execution; otherwise, we exit.

Part 4: The Tool Node (Action & Execution)

When the agent decides to invoke an external utility, the engine shifts execution control over to local system code.

In [ ]:
def from_scratch_tool_node(state: Dict[str, Any]) -> Dict[str, Any]:
    print("--- [NODE] Executing Tools ---")
    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", [])
    
    for tool_call in tool_calls:
        function_name = tool_call["name"]
        arguments = tool_call["args"]  # Pre-parsed by framework payload normalization
        
        print(f"Action: Invoking tool '{function_name}' with args {arguments}")
        result = tools_registry[function_name](**arguments)
        print(f"Observation: Result is {result}")
        
        observation_msg = HumanMessage(
            content=f"Observation from tool '{function_name}': The result is {result}. Now continue evaluating."
        )
        state["messages"].append(observation_msg)
        
    return state


• The Abstraction: Prebuilt components like ToolNode automatically execute functional maps and return complex data containers.
• The Reality: We keep a literal Python dictionary mapping string identifiers to raw, executable mathematical expressions (tools_registry = {"add": add}). The engine loops through the model's structural text instructions, extracts the arguments, triggers the local function using dictionary pointer lookups with argument unpacking (**arguments), formats the raw primitive integer results back into a HumanMessage string context block, and pushes it to the memory heap.

Part 5: The Loop Engine (The State Machine)

This is the primary orchestration loop that keeps the conversation running until a final solution is found.

In [ ]:
def run_agent(user_query: str):
    state = {"messages": [HumanMessage(content=user_query)]}
    
    max_iterations = 5
    for _ in range(max_iterations):
        state = agent_node(state)
        next_step = conditional_router(state)
        
        if next_step == "call_tools":
            state = from_scratch_tool_node(state)
        elif next_step == "end":
            print("\n--- Final Agent Response ---")
            print(state["messages"][-1].content)
            break


• The Abstraction: Compiling graphs with lines like app = graph.compile() masks the control system inside an external loop processor.
• The Reality: The entire execution engine is driven by a native Python for loop. When a tool finishes running, no special recursive handlers or extra functions are called. The block finishes execution, reaches the bottom boundary of the loop, and naturally ticks back to the top of the next iteration where agent_node(state) is triggered again with the modified thread history.

In [5]:
import json
from typing import Dict, Any, Callable
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

# Define native tools and mapping registers
def add(a: int, b: int) -> int: return a + b
tools_registry: Dict[str, Callable] = {"add": add}

# 1. Raw Schema Compilation (What .bind_tools does behind the scenes)
tools_schema = [{
    "name": "add",
    "description": "This tool adds two numbers together.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "integer"},
            "b": {"type": "integer"}
        },
        "required": ["a", "b"]
    }
}]

# Initialize the model with native schema filters
llm = ChatOllama(model="qwen2.5:3b", temperature=0).bind_tools(tools_schema)

# 2. Node A: The Pure Reasoning Wrapper
def agent_node(state: Dict[str, Any]) -> Dict[str, Any]:
    print("\n--- [NODE] Agent Thinking ---")
    response = llm.invoke(state["messages"])
    state["messages"].append(response)
    return state

# 3. Node B: The Native Python Conditional Router
def conditional_router(state: Dict[str, Any]) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print("-> Router Decision: Tool calls detected. Routing to TOOL NODE.")
        return "call_tools"
    print("-> Router Decision: Final answer reached. Routing to END.")
    return "end"

# 4. Node C: The Manual Tool Executor
def from_scratch_tool_node(state: Dict[str, Any]) -> Dict[str, Any]:
    print("--- [NODE] Executing Tools ---")
    last_message = state["messages"][-1]
    
    for tool_call in getattr(last_message, "tool_calls", []):
        function_name = tool_call["name"]
        arguments = tool_call["args"] # Pre-parsed by framework payload normalization
        
        print(f"Action: Invoking tool '{function_name}' with args {arguments}")
        result = tools_registry[function_name](**arguments)
        print(f"Observation: Result is {result}")
        
        # Append the execution outcome back into memory manually
        observation_msg = HumanMessage(
            content=f"Observation from tool '{function_name}': The result is {result}. Now continue evaluating."
        )
        state["messages"].append(observation_msg)
    return state

# 5. The Bare Loop Engine (Reaching Architecture Equality)
def run_agent(user_query: str):
    state = {
        "messages": [
            SystemMessage(content="You are a precise, tool-using assistant. Solve problems step-by-step."),
            HumanMessage(content=user_query)
        ]
    }
    
    max_iterations = 5
    for _ in range(max_iterations):
        state = agent_node(state)
        next_step = conditional_router(state)
        
        if next_step == "call_tools":
            state = from_scratch_tool_node(state)
        elif next_step == "end":
            print(f"\n--- Final Agent Response ---\n{state['messages'][-1].content}")
            break

run_agent("What is 3 plus 4?")



--- [NODE] Agent Thinking ---
-> Router Decision: Tool calls detected. Routing to TOOL NODE.
--- [NODE] Executing Tools ---
Action: Invoking tool 'add' with args {'a': 3, 'b': 4}
Observation: Result is 7

--- [NODE] Agent Thinking ---
-> Router Decision: Final answer reached. Routing to END.

--- Final Agent Response ---
The result of adding 3 and 4 together is 7. Is there anything else you would like to know or calculate?


In [10]:
import json
from typing import List, Dict, Any, Callable
from langchain_core.messages import HumanMessage, AIMessage
from langchain_ollama import ChatOllama

# 1. Define the raw tool functions
def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b

tools_registry: Dict[str, Callable] = {
    "add": add,
    "multiply": multiply
}

# 2. Simplified Schema format that ChatOllama explicitly looks for
tools_schema = [
    {
        "name": "add",
        "description": "Adds two integers together.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"}
            },
            "required": ["a", "b"]
        }
    },
    {
        "name": "multiply",
        "description": "Multiplies two integers together.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"}
            },
            "required": ["a", "b"]
        }
    }
]

# Initialize ChatOllama natively with tool bindings
llm = ChatOllama(model="qwen2.5:3b", temperature=0).bind_tools(tools_schema)

# 3. Custom Nodes and Graph Components

def agent_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Executes the LLM reasoning step and updates history."""
    print("\n--- [NODE] Agent Thinking ---")
    messages = state["messages"]
    response = llm.invoke(messages)
    state["messages"].append(response)
    return state


def from_scratch_tool_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Parses tool arguments, executes the local math, and formats the output."""
    print("--- [NODE] Executing Tools ---")
    last_message = state["messages"][-1]
    
    # FIX: Use ChatOllama's standardized top-level property (.tool_calls)
    tool_calls = getattr(last_message, "tool_calls", [])
    
    for tool_call in tool_calls:
        function_name = tool_call["name"]
        arguments = tool_call["args"]  # Already a clean python dict! No json.loads required.
        
        print(f"Action: Invoking tool '{function_name}' with args {arguments}")
        
        if function_name in tools_registry:
            result = tools_registry[function_name](**arguments)
        else:
            result = f"Error: Tool {function_name} not found."
            
        print(f"Observation: Result is {result}")
        
        # Append the context. The loop will pass this whole string to the agent next turn!
        observation_msg = HumanMessage(
            content=f"Observation from tool '{function_name}': The result is {result}. Now continue evaluating."
        )
        state["messages"].append(observation_msg)
        
    return state


def conditional_router(state: Dict[str, Any]) -> str:
    """Evaluates the state using ChatOllama's standard property structure."""
    last_message = state["messages"][-1]
    
    # FIX: Check if the model left behind valid tool calls in the standardized property
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print("-> Router Decision: Tool calls detected. Routing to TOOL NODE.")
        return "call_tools"
    
    print("-> Router Decision: Final answer reached. Routing to END.")
    return "end"


# 4. The Orchestration Loop (Manual Graph Cycle)
def run_agent(user_query: str):
    state = {
        "messages": [HumanMessage(content=user_query)]
    }
    
    max_iterations = 5
    for _ in range(max_iterations):
        # Step 1: Think (Agent Node)
        state = agent_node(state)
        
        # Step 2: Decide route (Conditional Node)
        next_step = conditional_router(state)
        
        if next_step == "call_tools":
            # Step 3: Act & Observe (Tool Node)
            state = from_scratch_tool_node(state)
            # NO NEED FOR EXTRA INVOKE LOGIC HERE: 
            # The loop automatically circles back up to Step 1 on the next iteration!
        elif next_step == "end":
            print("\n--- Final Agent Response ---")
            print(state["messages"][-1].content)
            break

# Run the query
run_agent("What is 3 plus 5? Take that result and multiply it by 10.")



--- [NODE] Agent Thinking ---
-> Router Decision: Tool calls detected. Routing to TOOL NODE.
--- [NODE] Executing Tools ---
Action: Invoking tool 'add' with args {'a': 3, 'b': 5}
Observation: Result is 8
Action: Invoking tool 'add' with args {'a': 8, 'b': 10}
Observation: Result is 18

--- [NODE] Agent Thinking ---
-> Router Decision: Tool calls detected. Routing to TOOL NODE.
--- [NODE] Executing Tools ---
Action: Invoking tool 'multiply' with args {'a': 8, 'b': 10}
Observation: Result is 80

--- [NODE] Agent Thinking ---
-> Router Decision: Final answer reached. Routing to END.

--- Final Agent Response ---
The multiplication of the previous result (8) by 10 resulted in 80. Therefore, the final answer is:

\( 8 \times 10 = 80 \)

This is the conclusion based on the operations you've asked for. If you have any more questions or need further calculations, feel free to ask!


Architecture Insights: The Unveiled Pipeline
By stripping away the layers of abstraction down to raw Python code, we expose exactly
how an agent interacts with code and memory:
●​ Memory Architecture: Memory isn't a complex entity; it is simply a list of
messages passed sequentially over local network sockets
(llm.invoke(state["messages"])). Keeping the SystemMessage at index 0
forces the model to honor its execution rules on every single iteration.
●​ Payload Normalization: When the framework converts data to .tool_calls, it
saves us from writing fragile custom regex code to strip text strings. It translates
raw model strings into structured JSON payloads that map cleanly to local
Python dictionaries.●​ Total Control Over the State Machine: When we use a bare for loop, there is
no magic thread scheduler. We control the boundary limits, manage error
feedback directly, and observe precisely how variables mutate step-by-step.
High-level agent wrappers are fantastic for building applications quickly. However,
understanding that they are ultimately just structured loops processing array states
gives you the technical depth to debug, optimize, and build robust custom AI systems.